# How common is NPM1 mutation in adult AML?

**Data kit lab** `tcga-aml-npm1-frequency` · Discovery · about 30 minutes

**The question.** In what share of the 200 adult de novo AML cases of the TCGA study is NPM1 mutated, and how many cases carry a mutation in each of the other genes listed in the paper's Table 1? Computed from the study's mutation file as served by the cBioPortal datahub, compared with the counts printed in the paper.

**What you do.** You read the public mutation file (MAF) and the sample list of the TCGA AML study, count mutated cases per gene the way a reader of the paper would have to, and check every number against Table 1 of the published report.

**The data.** cBioPortal study `laml_tcga_pub` (TCGA, NEJM 2013), from the cBioPortal Datahub on GitHub. Inside a company, the same kind of file sits in a cBioPortal instance or a genomics data store, and is queried through the portal's web page, its REST API, or a file like the one you read here. The data are cBioPortal's and TCGA's, used under cBioPortal's terms (ODbL, with attribution). Nothing is copied into this notebook: the first cell downloads the files from their source (about 0.9 MB, once).

**You need.** Python 3.10 or later with `pandas`.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'tcga-aml-npm1-frequency'
FILES = json.loads(r'''{
 "maf": {
  "url": "https://media.githubusercontent.com/media/cBioPortal/datahub/86690e1ed9752b1dcd50b5657f5f05eafa4b6b78/public/laml_tcga_pub/data_mutations.txt",
  "name": "maf.txt",
  "sha256": "2b6c6524fcb5974ea2b4850550591f5d0aa654ea27303126855795a6e45334f0",
  "bytes": 903277
 },
 "cases": {
  "url": "https://raw.githubusercontent.com/cBioPortal/datahub/86690e1ed9752b1dcd50b5657f5f05eafa4b6b78/public/laml_tcga_pub/case_lists/cases_sequenced.txt",
  "name": "cases.txt",
  "sha256": "b2956719cb9f3d18cc729e54985da0e2e9f61b16cb3769d0db578a22ec952050",
  "bytes": 3445
 }
}''')
PUBLISHED = json.loads(r'''{
 "n_cases": {
  "label": "Cases with mutation data",
  "value": 200,
  "tolerance": {
   "exact": true
  }
 },
 "npm1_n": {
  "label": "NPM1 mutated, cases",
  "value": 54,
  "tolerance": {
   "exact": true
  }
 },
 "flt3_n": {
  "label": "FLT3 mutated, cases",
  "value": 56,
  "tolerance": {
   "exact": true
  }
 },
 "dnmt3a_n": {
  "label": "DNMT3A mutated, cases",
  "value": 51,
  "tolerance": {
   "exact": true
  }
 },
 "idh1_or_idh2_n": {
  "label": "IDH1 or IDH2 mutated, cases",
  "value": 39,
  "tolerance": {
   "exact": true
  }
 },
 "nras_or_kras_n": {
  "label": "NRAS or KRAS mutated, cases",
  "value": 23,
  "tolerance": {
   "exact": true
  }
 },
 "runx1_n": {
  "label": "RUNX1 mutated, cases",
  "value": 19,
  "tolerance": {
   "exact": true
  }
 },
 "tet2_n": {
  "label": "TET2 mutated, cases",
  "value": 17,
  "tolerance": {
   "exact": true
  }
 },
 "tp53_n": {
  "label": "TP53 mutated, cases",
  "value": 16,
  "tolerance": {
   "exact": true
  }
 },
 "cebpa_n": {
  "label": "CEBPA mutated, cases",
  "value": 13,
  "tolerance": {
   "exact": true
  }
 },
 "wt1_n": {
  "label": "WT1 mutated, cases",
  "value": 12,
  "tolerance": {
   "exact": true
  }
 },
 "ptpn11_n": {
  "label": "PTPN11 mutated, cases",
  "value": 9,
  "tolerance": {
   "exact": true
  }
 },
 "kit_n": {
  "label": "KIT mutated, cases",
  "value": 8,
  "tolerance": {
   "exact": true
  }
 },
 "npm1_pct": {
  "label": "NPM1 mutated, % of cases",
  "value": 27.0,
  "tolerance": {
   "abs": 0.5
  }
 }
}''')
PUBLISHED_FROM = 'PMC3767041 full-text XML (no page numbers; the article as a whole is pages 2059 to 2074): Table 1, "Characteristics of the 200 Patients", block "Mutation — no./total no. (%)", rows NPM1 to KIT; abstract, NPM1 percentage | https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pmc&id=3767041&retmode=xml'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. Who is in the study? (the case list)

A cBioPortal "case list" is a plain text file: a few `key: value` header lines, then one line `case_list_ids:` followed by the sample ids, tab-separated. This lab's case list is "Samples with mutation data" — the denominator for every percentage in Table 1.

In [ ]:
text = data_file("cases").read_text()
ids = text.split("case_list_ids:")[1].split()
print("sample ids in the case list:", len(ids), "| distinct:", len(set(ids)))
n_cases = len(set(ids))
assert n_cases == 200, "the case list no longer holds 200 distinct ids"


## 2. The mutation calls (the MAF)

A MAF (Mutation Annotation Format) file holds one row per called mutation in one sample. `Hugo_Symbol` names the gene, `Tumor_Sample_Barcode` the sample, `Variant_Classification` the kind of change (missense, nonsense, splice site, ...), and `HGVSp_Short` the protein change.

In [ ]:
import pandas as pd

maf = pd.read_csv(data_file("maf"), sep="\t", low_memory=False)
print("shape:", maf.shape)
print(maf[["Hugo_Symbol", "Tumor_Sample_Barcode", "Variant_Classification", "HGVSp_Short"]].head(3).to_string(index=False))
print()
print(maf["Variant_Classification"].value_counts().to_string())


## 3. The denominator trap

The case list has 200 ids. How many of them actually appear in the MAF?

In [ ]:
in_maf = set(maf["Tumor_Sample_Barcode"])
print("distinct samples with at least one mutation row:", len(in_maf))
missing = sorted(set(ids) - in_maf)
print("case-list samples with no row in the MAF:", len(missing))
print(missing)

assert set(maf["Tumor_Sample_Barcode"]).issubset(set(ids)), "a MAF sample is not in the case list"


Four of the 200 listed cases have no mutation row at all. A case with no called mutation is still a case: the denominator for every percentage in Table 1 is the case list (200), not the set of samples that happen to appear in the MAF (196). Dividing by 196 instead of 200 would move NPM1 from 27.0% to 27.6% — close enough to look right and wrong all the same.

We do **not** filter the MAF on `Variant_Classification`, `Variant_Type` or anything else: every validated row counts (checked in the next section).

## 4. One gene, counted properly

NPM1 first. A sample can have more than one mutation row in the same gene; the table counts **samples**, not rows.

In [ ]:
npm1_rows = maf[maf["Hugo_Symbol"] == "NPM1"]
npm1_samples = set(npm1_rows["Tumor_Sample_Barcode"])
print("NPM1 rows:", len(npm1_rows), "| distinct samples:", len(npm1_samples))


55 rows, 54 distinct samples: one sample carries two separate NPM1 mutations and must only be counted once.

## 5. All twelve rows of the "Mutation" block

Table 1 lists twelve genes (an "or" row takes the union of the two genes' sample sets).

In [ ]:
GENE_ROWS = [
    ("NPM1", ["NPM1"]), ("FLT3", ["FLT3"]), ("DNMT3A", ["DNMT3A"]),
    ("IDH1 or IDH2", ["IDH1", "IDH2"]), ("NRAS or KRAS", ["NRAS", "KRAS"]),
    ("RUNX1", ["RUNX1"]), ("TET2", ["TET2"]), ("TP53", ["TP53"]),
    ("CEBPA", ["CEBPA"]), ("WT1", ["WT1"]), ("PTPN11", ["PTPN11"]), ("KIT", ["KIT"]),
]

table = []
gene_n = {}
for label, genes in GENE_ROWS:
    samples = set(maf.loc[maf["Hugo_Symbol"].isin(genes), "Tumor_Sample_Barcode"])
    n = len(samples)
    gene_n[label] = n
    table.append({"row": label, "cases": n, "pct_of_200": round(100 * n / n_cases, 1)})

print(pd.DataFrame(table).to_string(index=False))


## 6. The IDH1-or-IDH2 union, in detail

This is information only, not one of the checks below: it shows why an "or" row is not the sum of the two single-gene counts.

In [ ]:
idh1 = set(maf.loc[maf["Hugo_Symbol"] == "IDH1", "Tumor_Sample_Barcode"])
idh2 = set(maf.loc[maf["Hugo_Symbol"] == "IDH2", "Tumor_Sample_Barcode"])
print("IDH1 alone:", len(idh1), "| IDH2 alone:", len(idh2),
      "| in both:", len(idh1 & idh2), "| union:", len(idh1 | idh2), "| sum:", len(idh1) + len(idh2))


## 7. NPM1 as a percentage, and the file's overall size

The published percentage is not rounded by this notebook: it is `100 * npm1_n / n_cases`, kept as a float, and compared with the paper's printed "27" within half a point.

In [ ]:
npm1_n = gene_n["NPM1"]
npm1_pct = 100 * npm1_n / n_cases
print(f"NPM1: {npm1_n} of {n_cases} cases = {npm1_pct:.2f}%")

print()
print("rows in this MAF:", len(maf))
print("the paper reports 2,585 validated tier 1 mutations (2,315 SNVs and 270 indels) — information only, not a check.")


## 8. Check against the published table

`PUBLISHED` holds the fourteen values copied by code from Table 1 of the paper. `report` prints your value beside each one.

In [ ]:
RESULTS = {"n_cases": int(n_cases)}
ID_BY_LABEL = {
    "NPM1": "npm1", "FLT3": "flt3", "DNMT3A": "dnmt3a", "IDH1 or IDH2": "idh1_or_idh2",
    "NRAS or KRAS": "nras_or_kras", "RUNX1": "runx1", "TET2": "tet2", "TP53": "tp53",
    "CEBPA": "cebpa", "WT1": "wt1", "PTPN11": "ptpn11", "KIT": "kit",
}
for label, n in gene_n.items():
    RESULTS[f"{ID_BY_LABEL[label]}_n"] = int(n)
RESULTS["npm1_pct"] = float(npm1_pct)

report(RESULTS)


## What the result says

Nine of the twelve gene counts (and the NPM1 percentage) reproduce exactly from this public file. Three do not: DNMT3A, RUNX1 and IDH1-or-IDH2 come out one to two cases short of the published count. The paper's supplementary material was not opened for this lab, so the cause is not established here — only that the file's own total (2,532 mutation rows) is smaller than the paper's reported 2,585 validated tier-1 mutations, so some calls in the paper are not in this public file.

## Try next

1. Drop every row classed `Splice_Region` and recompute FLT3 and TP53. Which published counts break, and why does the lab's rule keep them in?
2. Look up one of the four case-list samples with no MAF row (listed in section 3) on the cBioPortal website. What does the portal show for that sample's mutation data?
3. Compute the same twelve counts over the 196 samples that actually appear in the MAF, instead of the 200-sample case list. How much does NPM1's percentage move?